In [ ]:
import gensim.downloader as api
from numpy import dot
from numpy.linalg import norm

# Step 1: Load pre-trained Word2Vec embeddings
print("Downloading pre-trained Word2Vec embeddings...")
word2vec = api.load('word2vec-google-news-300')  # Loads a smaller version of Google News Word2Vec
print("Download completed!")

# Step 2: Define a function to calculate cosine similarity
def cosine_similarity(word1, word2):
    try:
        vec1 = word2vec[word1]
        vec2 = word2vec[word2]
        similarity = dot(vec1, vec2) / (norm(vec1) * norm(vec2))
        return similarity
    except KeyError as e:
        return f"Error: {e}. One or both words not found in the vocabulary."

# Step 3: Test the function with example words
word1 = "king"
word2 = "queen"
similarity = cosine_similarity(word1, word2)
print(f"Cosine Similarity between '{word1}' and '{word2}': {similarity}")


[==================================================] 100.0% 1662.8/1662.8MB downloaded
Download completed!
Cosine Similarity between 'king' and 'queen': 0.6510956287384033


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

# Step 1: Define the Encoder
encoder_inputs = Input(shape=(None, 256))  # Replace 256 with your input dimension
encoder_lstm = LSTM(256, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

# Step 2: Define the Decoder
decoder_inputs = Input(shape=(None, 256))  # Replace 256 with your input dimension
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(5000, activation='softmax')  # Replace 5000 with your target vocabulary size
decoder_outputs = decoder_dense(decoder_outputs)

# Step 3: Compile the Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(model.summary())



In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout

# Step 1: Define sample text data and labels
text_data = [
    "I love this product. It is amazing!",
    "This is the worst experience I have ever had.",
    "The quality of the item is excellent.",
    "I am very disappointed with the service.",
    "Fantastic! Highly recommend this to everyone.",
    "Not worth the money. Completely dissatisfied."
]
labels = [1, 0, 1, 0, 1, 0]  # 1 = Positive, 0 = Negative sentiment

# Step 2: TF-IDF Features
vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = vectorizer.fit_transform(text_data).toarray()

# Step 3: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, labels, test_size=0.2, random_state=42)

# Step 4: Build LSTM Model
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=X_tfidf.shape[1]))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))  # Binary sentiment classification
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Step 5: Train the model
model.fit(X_train, np.array(y_train), batch_size=32, epochs=5, validation_data=(X_test, np.array(y_test)))

# Step 6: Evaluate the model
loss, accuracy = model.evaluate(X_test, np.array(y_test))
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")


